In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path
import itertools
from functools import reduce
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

In [ ]:
results_path = Path("./results/")

In [ ]:
new_dir = Path("./transformed_path")

In [ ]:
for file in glob.glob(os.path.join(results_path, "*.csv")):
    file
    os.makedirs(new_dir, exist_ok=True)
    file = Path(file)
    if file.stem in ["DecisionTree_summary_results", "MLP_summary_results", "SVM_summary_results"] :
        continue
    df = pd.read_csv(file)
    df = df[["isic_id", "prob_class_0",]]
    
    name = file.stem.split("_")[0]
    df = df.rename(columns={
        col: f"{col}_{name}" for col in df.columns if col not in ["isic_id"]
    })
    
    df = df.to_csv(f"{os.path.join(new_dir, name)}.csv",index=False)

In [ ]:
to_fuse = {}
df = pd.read_csv(file)
df_labels = df[["isic_id", "true_label"]]


for file in glob.glob(os.path.join(new_dir, "*.csv")):
    df = pd.read_csv(file)
    df = df.iloc[:, :3]
    file_name = Path(file).stem
    to_fuse[file_name] = df
    #df = df[["prob_class_0", "prob_class_1"]]
    # file_name = file_name.split("_")[0]
    # to_fuse[file_name] = df


In [ ]:
to_fuse

In [ ]:
nomes_extratores = list(to_fuse.keys())

In [ ]:
save_path = Path("Combinações")
save_path.mkdir(exist_ok=True)

print("Iniciando processo de fusão...")
for r in range(2, len(nomes_extratores) + 1):
    for combo_nomes in itertools.combinations(nomes_extratores, r):
        dfs_to_merge = [to_fuse[nome] for nome in combo_nomes]

        merged_df = reduce(lambda left, right: pd.merge(left, right, on='isic_id', how='outer'), dfs_to_merge)
        merged_df = pd.merge(left=merged_df, right=df_labels, how="inner", on="isic_id")

        output_filename = f'{"_".join(combo_nomes)}.csv'
        merged_df.to_csv(os.path.join(save_path, output_filename), index=False)
        print(f'--> Arquivo gerado: {output_filename}')

In [ ]:
fused_outputs = Path("Combinações")

In [ ]:
# Voto Majoritário

In [ ]:
def major_count(df, to_ignore):
    prob_cols = df.columns.difference(to_ignore)
    
    limiar = 0.5
    qtd_cols = len(prob_cols)
    metade_qtd_cols = qtd_cols/2
    
    votos_classe_0 = (df[prob_cols] > limiar).sum(axis=1)
    conditions = [
        votos_classe_0 > metade_qtd_cols,
        votos_classe_0 < metade_qtd_cols
    ]
    
    escolhas = [0, 1]
    
    media_geral_probs = df[prob_cols].mean(axis=1)
    
    desempate = np.where(media_geral_probs > limiar, 0, 1)
    
    df['major_vote_class'] = np.select(conditions, escolhas, default=desempate)
    return df

In [ ]:
# Regra da Soma

In [ ]:
def sum_rule(df, to_ignore):
    prob_cols = df.columns.difference(to_ignore)
    limiar = 0.5

    media_geral_probs = df[prob_cols].mean(axis=1)

    df['sum_rule_class'] = np.where(media_geral_probs > limiar, 0, 1)
    return df

In [ ]:
# Regra do Produto

In [ ]:
def product_rule(df, to_ignore):
    prob_cols = df.columns.difference(to_ignore)
    limiar = 0.5

    prob_0 = df[prob_cols].prod(axis=1)
    prob_1 = 1 - df[prob_cols].prod(axis=1)

    df['product_rule_class'] = np.where(prob_0 > prob_1, 0, 1)
    return df

In [ ]:
def apply_fusion_rules(save_path, fusion_function, fused_outputs, rule_name_output):

    os.makedirs(save_path, exist_ok=True)
    to_ignore = ["isic_id", "true_label"]
    
    summary_dict = {}
    
    for file in glob.glob(os.path.join(fused_outputs, "*.csv")):
    
        fusion_name = Path(file).name
        
        df = pd.read_csv(file)
        df = fusion_function(df, to_ignore)
    
        df.to_csv(f"{save_path}/{fusion_name}.csv", index=False)
        
        
        acc = accuracy_score(df["true_label"], df[rule_name_output])
        f1_macro = f1_score(df["true_label"], df[rule_name_output], average="macro")
        recall = recall_score(df["true_label"], df[rule_name_output])
        precision = precision_score(df["true_label"], df[rule_name_output])
        summary_dict[fusion_name] = [acc, f1_macro, recall, precision]
        
        
    df_summary = pd.DataFrame.from_dict(summary_dict, orient="index", columns=["accuracy", "f1_macro", "recall", "precision"])
    df_summary.sort_values("f1_macro", ascending=False, inplace=True)
    df_summary.to_csv(os.path.join(save_path,f"{rule_name_output}_sumario_result.csv"), index=False)
    return df_summary

In [ ]:
save_vote = Path("Regra_Voto_Resultados")

summary = apply_fusion_rules(save_vote, major_count, fused_outputs, "major_vote_class")

In [ ]:
summary

In [ ]:
save_sum = Path("Regra_Sum_Resultados")

summary = apply_fusion_rules(save_sum, sum_rule, fused_outputs, "sum_rule_class")

In [ ]:
summary

In [ ]:
save_prod = Path("Regra_Product_Resultados")

summary = apply_fusion_rules(save_prod, product_rule, fused_outputs, "product_rule_class")

In [ ]:
summary.head(10)

In [ ]:
major_vote_path = Path("Regra_Voto_Resultados")
os.makedirs(major_vote_path, exist_ok=True)
to_ignore = ["isic_id", "true_label"]

summary_dict = {}

for file in glob.glob(os.path.join(fused_outputs, "*.csv")):

    fusion_name = Path(file).name
    
    df = pd.read_csv(file)
    df = major_count(df, to_ignore)

    df.to_csv(f"{major_vote_path}/{fusion_name}.csv", index=False)
    
    
    acc = accuracy_score(df["true_label"], df["major_vote_class"])
    f1_macro = f1_score(df["true_label"], df["major_vote_class"], average="macro")
    recall = recall_score(df["true_label"], df["major_vote_class"])
    summary_dict[fusion_name] = [acc, f1_macro, recall]
    
    
df_summary = pd.DataFrame.from_dict(summary_dict, orient="index", columns=["accuracy", "f1_macro", "recall"])
df_summary.to_csv(os.path.join(major_vote_path,"sumario_result.csv"), index=False)

In [ ]:
df_summary